<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [14]</a>'.</span>

# Fine-tuning YOLOP for BeamNG control correction, using Comma2k19

This notebook fine-tunes the project's YOLOP model (the official `YOLOP/lib/models/yolop.py`
architecture, pretrained weights in `YOLOP/weights/End-to-end.pth`) so that, given a dashcam
frame plus the car's *current* throttle/brake/steer/speed, it predicts *corrected*
**steer and brake only** to drive in BeamNG.

Throttle/acceleration is intentionally not predicted here -- it is handled by a separate,
non-adaptive cruise-control routine outside this model (see
`2 Closed Loop BeamNG Control.ipynb`). The previous 3-output version of this model (throttle,
brake, steer sharing one fusion head) let comma2k19's emergency-braking frames teach the
network a spurious "max brake -> wheel centered" shortcut, which collapsed steering whenever
BeamNG's bridge/overpass shadows triggered the brake output. `YOLOPControlNet` now uses
separate `steer_head`/`brake_head` modules with a gradient stop on the brake path (see
`control_finetune.py`) so a brake decision can no longer push the shared features that drive
steering.

The YOLOP backbone is kept frozen (same spirit as the LoRA fine-tuning in the original version
of this notebook: keep the big pretrained model frozen, train only a small new part). A small
shared trunk + two heads are trained on top, using [Comma2k19](https://github.com/commaai/comma2k19) —
a real-world driving dataset — as the source of (image, telemetry) -> (future telemetry)
examples. The training split is also passed through aggressive photometric augmentations
(synthetic shadows, Cutout, brightness/contrast/color jitter -- see
`control_finetune.build_train_augmentations`) so the model doesn't key off comma2k19's
particular lighting/shadow conditions as a shortcut for "should I brake."

Comma2k19 has no explicit brake/gas pedal channel, so brake is derived from the IMU's
longitudinal acceleration and converted into BeamNG's `brake` (0..1) via
`comma_beamng_transforms.py`. Steering wheel angle (degrees) is converted to BeamNG's
`steering` (-1..1) the same way. These transform functions are written once and are directly
reusable by `yolop_beamng.py` later.

Pipeline: setup -> download a small Comma2k19 subset -> build a (frame, telemetry) manifest ->
load the frozen YOLOP base model -> train the new steer/brake heads -> evaluate -> sanity-check
inference.

In [1]:
def _torch_cuda_ready():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

if _torch_cuda_ready():
    print("torch+CUDA already installed and working -- skipping the uninstall/reinstall below "
          "(avoids re-downloading multi-GB wheels before an unattended overnight run).")
else:
    # Plain `pip install torch` on Windows resolves to the CPU-only wheel. Install the
    # CUDA build explicitly so torch.cuda.is_available() picks up the local GPU. cu126
    # works with any driver supporting CUDA >= 12.6 (check with `nvidia-smi`); bump the
    # index URL (e.g. cu128, cu129) if pip can't find a wheel for your Python version.
    get_ipython().run_line_magic("pip", "uninstall -y torch torchvision torchaudio -q")
    get_ipython().run_line_magic("pip", "install -q torch torchvision --index-url https://download.pytorch.org/whl/cu126")

get_ipython().run_line_magic("pip", "install -q opencv-python imageio-ffmpeg pandas numpy albumentations")

# Optional: only needed for tools/download_comma2k19.py's automated subset download.
# If it fails to install (common on Windows), the download script falls back to
# printing manual instructions for a regular BitTorrent client.
get_ipython().run_line_magic("pip", "install -q libtorrent")

torch+CUDA already installed and working -- skipping the uninstall/reinstall below (avoids re-downloading multi-GB wheels before an unattended overnight run).


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\nalli\\Documents\\Workspace Uni\\TP FINAL Vision\\TP-final-Vision-Artificial\\.venv\\Lib\\site-packages\\cv2\\cv2.pyd'
Check the permissions.



Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement libtorrent (from versions: none)
ERROR: No matching distribution found for libtorrent


In [2]:
# `albumentations`/`albucore` (installed above) declare `opencv-python-headless` as a
# dependency. It ships into the same `cv2` import namespace as the `opencv-python` installed
# just before it in the same command -- pip doesn't treat them as conflicting, so whichever
# one's files land last in site-packages silently wins. If headless wins, `cv2.imshow` /
# `cv2.waitKey` / `cv2.destroyAllWindows` start raising "function not implemented... GTK+/Cocoa"
# -- which breaks `2 Closed Loop BeamNG Control.ipynb`'s debug window, with BeamNG itself still
# opening fine (it doesn't go through cv2's GUI backend). This has already happened twice this
# session purely from re-running the cell above. Detect and self-heal every time this notebook
# runs, instead of relying on remembering to fix it by hand again.
import subprocess
import sys

try:
    import cv2
    import numpy as _np
    cv2.imshow("_gui_check", _np.zeros((2, 2, 3), dtype=_np.uint8))
    cv2.waitKey(1)
    cv2.destroyAllWindows()
    print("[opencv-gui-check] cv2 GUI backend OK.")
except Exception as exc:
    print(f"[opencv-gui-check] cv2 GUI backend broken ({exc!r}) -- "
          "opencv-python-headless likely clobbered opencv-python. Reinstalling opencv-python...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "opencv-python-headless"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                     "opencv-python==4.13.0.92"], check=True)
    print("[opencv-gui-check] Reinstalled opencv-python -- restart the kernel before using cv2 "
          "GUI functions in this same process (the broken .pyd is already loaded in memory).")

[opencv-gui-check] cv2 GUI backend OK.


In [3]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
device = "cuda" if torch.cuda.is_available() else "cpu"

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti


## 1. Download a small Comma2k19 subset

The full dataset is ~100GB over BitTorrent. The torrent itself only contains 10 whole-chunk
zip files (`Chunk_1.zip` .. `Chunk_10.zip`, ~9-10GB each) -- BitTorrent can only select whole
files, so there's no way to fetch individual segments over the wire despite the per-segment
folder layout documented in the upstream README (that layout describes what's *inside* each
zip, not the torrent's own file list).

So `tools/download_comma2k19.py` downloads exactly one whole chunk zip, then extracts only
`--max-segments` segment folders from inside it locally (without unpacking the other ~200
segments in the same chunk), and deletes the zip afterward to reclaim the ~9GB.

It tries `libtorrent` first, then falls back to driving a running **qBittorrent**'s Web UI
(via `qbittorrent-api`) -- only this fallback worked in this environment, since `libtorrent`
has no wheel for Windows + Python 3.14. One-time setup if you haven't already:

1. Install qBittorrent (already done: `winget install qBittorrent.qBittorrent`) and make sure
   it's actually **running** (it doesn't auto-start).
2. Open it once, go to **Tools > Options > Web UI**, check "Web User Interface", note the
   port (default 8080), and set a username/password.
3. Set `QBT_USERNAME` / `QBT_PASSWORD` below to match.

If qBittorrent isn't reachable either, the script prints manual instructions and exits with
an error instead of silently leaving `RAW_DIR` empty.

In [4]:
import os
from pathlib import Path

RAW_DIR = "data/comma2k19/raw"

n_segments = len(list(Path(RAW_DIR).rglob("video.hevc")))

if n_segments == 0:
    os.environ["QBT_USERNAME"] = "admin"      # match what you set in qBittorrent's Web UI options
    os.environ["QBT_PASSWORD"] = "CHANGE_ME"  # do not commit a real password into the notebook

    !python tools/download_comma2k19.py --out {RAW_DIR} --chunks 1 --max-segments 15

    n_segments = len(list(Path(RAW_DIR).rglob("video.hevc")))
else:
    # Segments already present (e.g. chunks 1-10 downloaded ahead of time via
    # tools/download_comma2k19.py run separately) -- skip re-triggering qBittorrent.
    print(f"{n_segments} segment(s) already present in {RAW_DIR}, skipping download.")

assert n_segments > 0, (
    f"No segments found in {RAW_DIR} -- check the error/instructions printed above "
    "(likely qBittorrent Web UI credentials or connectivity) before continuing."
)
print(f"\n{n_segments} segment(s) ready in {RAW_DIR}")

150 segment(s) already present in data/comma2k19/raw, skipping download.

150 segment(s) ready in data/comma2k19/raw


## 2. Inspect one segment's `processed_log` layout

comma2k19 mirrors have varied slightly in exact `processed_log` sub-paths (e.g.
`CAN/car_speed` vs `CAN/speed`). Before building the full manifest, inspect one downloaded
segment to confirm the signal lookup in `tools/prepare_comma2k19_control_dataset.py`
(fuzzy-matched by keyword tokens) is actually finding the speed/steering/IMU arrays. If it
isn't, adjust `SIGNAL_TOKENS` in that script to match what's printed below.

In [5]:
segments = list(Path(RAW_DIR).rglob("video.hevc"))
assert segments, f"No video.hevc found under {RAW_DIR} -- re-run the download cell above first."

first_segment = segments[0].parent
print(f"Inspecting: {first_segment}\n")

!python tools/prepare_comma2k19_control_dataset.py --inspect "{first_segment}"

Inspecting: data\comma2k19\raw\Chunk_9\99c94dc769b5d96e_2018-10-12--20-13-54\10



Traceback (most recent call last):
  File "C:\Users\nalli\Documents\Workspace Uni\TP FINAL Vision\TP-final-Vision-Artificial\tools\prepare_comma2k19_control_dataset.py", line 35, in <module>
    import numpy as np
ModuleNotFoundError: No module named 'numpy'


## 3. Build the training manifest

Extracts frames from each segment's `video.hevc`, aligns speed/steering/IMU-acceleration to
each frame and to a future horizon (`--horizon-seconds`, default 1.0s), applies the
comma2k19 -> BeamNG transforms, and writes `data/comma2k19/manifest.csv` with a per-segment
train/dev/test split (70/15/15) — split by segment, not by frame, so adjacent near-duplicate
frames don't leak across splits.

In [6]:
FRAMES_DIR = "data/comma2k19/frames"
MANIFEST_PATH = "data/comma2k19/manifest.csv"

# Skip if already built (mirrors section 1's download skip-guard) -- extract_frames below
# always re-runs ffmpeg over every segment's video.hevc with no skip-if-exists check of its
# own, so re-running this cell unguarded would re-decode all ~150 segments' video again for no
# reason whenever the notebook is re-run top-to-bottom (e.g. for a later training run that
# only needs sections 4+).
if Path(MANIFEST_PATH).exists():
    print(f"{MANIFEST_PATH} already exists, skipping rebuild.")
else:
    !python tools/prepare_comma2k19_control_dataset.py --raw-dir {RAW_DIR} --frames-dir {FRAMES_DIR} --manifest {MANIFEST_PATH} --horizon-seconds 1.0

data/comma2k19/manifest.csv already exists, skipping rebuild.


In [7]:
import pandas as pd

manifest = pd.read_csv(MANIFEST_PATH)
print(manifest["split"].value_counts())
print()
print(manifest[["throttle_t", "brake_t", "steer_t", "speed_t",
                "throttle_target", "brake_target", "steer_target"]].describe())

split
train    122526
test      27045
dev       25578
Name: count, dtype: int64

          throttle_t        brake_t        steer_t        speed_t  \
count  175149.000000  175149.000000  175149.000000  175149.000000   
mean        0.048115       0.053150      -0.000784      21.859185   
std         0.109161       0.063794       0.030159      10.685079   
min         0.000000       0.000000      -1.000000       0.000000   
25%         0.000000       0.000000      -0.003843      15.065278   
50%         0.000000       0.034906       0.000000      25.790972   
75%         0.041621       0.084511       0.004139      30.037500   
max         1.000000       0.836012       0.620795      39.871528   

       throttle_target   brake_target   steer_target  
count    175149.000000  175149.000000  175149.000000  
mean          0.048121       0.053182      -0.000796  
std           0.109231       0.063835       0.029920  
min           0.000000       0.000000      -1.000000  
25%           0.000000

## 4. Load the frozen YOLOP base model + steer/brake heads

`YOLOPControlNet` (in `control_finetune.py`) loads `YOLOP/lib/models/yolop.py` with
`YOLOP/weights/End-to-end.pth` (same pattern as `yolop_reproduction.ipynb`), freezes every
backbone parameter, and adds a small trainable shared trunk that fuses pooled image features
with the current `[throttle, brake, steer, speed]` telemetry, feeding two separate heads:
`steer_head` (tanh, -1..1) and `brake_head` (sigmoid, 0..1). `brake_head` reads a
`.detach()`-ed copy of the shared trunk's output (`detach_brake_grad=True`, the default), so
the brake loss can only update `brake_head` itself -- it never reaches the shared
trunk/numeric branch that `steer_head` also depends on.

In [8]:
from control_finetune import YOLOPControlNet

control_model = YOLOPControlNet(device=device).to(device)

n_trainable = sum(p.numel() for p in control_model.trainable_parameters())
n_frozen = sum(p.numel() for p in control_model.backbone.parameters())
print(f"Trainable (shared trunk + steer/brake heads + numeric branch): {n_trainable:,}")
print(f"Frozen (YOLOP backbone):                                      {n_frozen:,}")

Trainable (shared trunk + steer/brake heads + numeric branch): 94,274
Frozen (YOLOP backbone):                                      7,940,846


## 4b. Precompute hard road/lane/vehicle-masked frames (one-time)

`lane_mask.py`'s `full_road_mask` (drivable area OR lane line OR a detected vehicle,
thresholded -- the same signal `YOLOPControlNet`'s internal `_road_mask`/`_masked_pool`
already uses, just applied as a hard *pixel* mask before the frame reaches the backbone,
instead of a soft post-pooling weight) is baked into a parallel frame directory here, once,
rather than inside `Comma2k19ControlDataset.__getitem__`. Masking needs a full frozen-backbone
forward pass per image (`segment_full_res`); the training loop already pays for one backbone
forward per batch, so doing a second one on the fly in `__getitem__` would double that cost on
every epoch, every sample, for no reason -- this way it's paid once, ever, per frame.

Deliberately *not* using `fit_lane_curves`/`road_mask_with_lane_attention` (the quadratic
lane-curve fit) here -- the curve fit has turned out unreliable enough in practice that it's
not trustworthy as a training signal. Training instead matches the closed loop's plain "road"
mask mode (drivable area + lane line + vehicles, no curve math), which is also the simpler and
more robust of the two.

Idempotent: frames already present in `MASKED_FRAMES_DIR` are skipped, so interrupting and
re-running this cell is safe.

In [9]:
MASKED_FRAMES_DIR = "data/comma2k19/frames_masked"

!python tools/precompute_masked_frames.py --manifest {MANIFEST_PATH} --frames-dir {FRAMES_DIR} --out-dir {MASKED_FRAMES_DIR} --log-every 2000

n_masked = sum(1 for _ in Path(MASKED_FRAMES_DIR).rglob("*.jpg"))
print(f"{n_masked} masked frames present in {MASKED_FRAMES_DIR}")

Traceback (most recent call last):
  File "C:\Users\nalli\Documents\Workspace Uni\TP FINAL Vision\TP-final-Vision-Artificial\tools\precompute_masked_frames.py", line 32, in <module>
    import cv2
ModuleNotFoundError: No module named 'cv2'


175149 masked frames present in data/comma2k19/frames_masked


## 5. Datasets and dataloaders

In [10]:
import os

import torch
from torch.utils.data import DataLoader
from control_finetune import Comma2k19ControlDataset

# `augment` defaults to True for the train split (and False for dev/test) -- see
# Comma2k19ControlDataset. Dev/test must stay clean (no synthetic shadows/Cutout) to measure
# real generalization, not augmentation-time noise.
#
# Pointed at MASKED_FRAMES_DIR (not FRAMES_DIR): every frame here already had
# lane_mask.full_road_mask applied once, in section 4b, so __getitem__ doesn't need to change
# at all -- it's still a plain cv2.imread + preprocess_image_bgr, just reading already-masked
# pixels instead of the raw frame.
train_ds = Comma2k19ControlDataset(MANIFEST_PATH, MASKED_FRAMES_DIR, split="train")
dev_ds = Comma2k19ControlDataset(MANIFEST_PATH, MASKED_FRAMES_DIR, split="dev")
test_ds = Comma2k19ControlDataset(MANIFEST_PATH, MASKED_FRAMES_DIR, split="test")

BATCH_SIZE = 64  # bumped from 16 -- the frozen-backbone forward leaves plenty of VRAM headroom
                 # (benchmarked: 3.4GB/16GB peak), CPU-side decode/augment was the real bottleneck

# NUM_WORKERS=6, not os.cpu_count()-2 (10 on this machine): this machine only has ~16GB system
# RAM. train_loader/dev_loader/test_loader all use persistent_workers=True, and run_epoch
# alternates train/dev each epoch, so once both have iterated once their worker pools (and
# test_loader's, once section 7 runs) stay alive *concurrently* -- 3 x num_workers processes,
# not just num_workers. At num_workers=8-10 this reliably hit
# `CUDA error: out of memory` inside the *pin_memory* thread (a Windows page-locked-host-memory
# quota, not GPU VRAM -- nvidia-smi showed >10GB free at the time). pin_memory is therefore
# left off entirely (the H2D transfer isn't the bottleneck here anyway), and num_workers=6 was
# the largest value that survived a simulated train->dev->test lifecycle with all three loaders'
# workers alive at once (benchmarked: ~4x throughput over the num_workers=0 baseline, settling
# around 2GB free RAM rather than OOMing).
NUM_WORKERS = 6

# Each DataLoader worker process inherits PyTorch's default intra-op thread count, which tries
# to use every core *per worker* -- with NUM_WORKERS>1 that oversubscribes the machine well
# past its core count, and workers end up fighting each other instead of the main process
# actually waiting on them. Cap the main process's own intra-op threads since the heavy lifting
# (image decode/augmentation) now happens in the worker processes, not here.
if NUM_WORKERS > 0:
    torch.set_num_threads(1)

loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=False,
    persistent_workers=NUM_WORKERS > 0,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

print(f"train={len(train_ds)} dev={len(dev_ds)} test={len(test_ds)}")
print(f"train augmentations active: {train_ds.augmentations is not None}")
print(f"dev augmentations active:   {dev_ds.augmentations is not None}")
print(f"batch_size={BATCH_SIZE} num_workers={NUM_WORKERS}")

train=122526 dev=25578 test=27045
train augmentations active: True
dev augmentations active:   False
batch_size=64 num_workers=6


## 6. Train the steer/brake heads

Plain PyTorch training loop over only the unfrozen parameters (`control_model.trainable_parameters()`).
Loss is `SmoothL1Loss` per output channel, with steering weighted higher since it's more
safety-critical than brake magnitude. The brake channel's gradient is stopped before the
shared trunk (`detach_brake_grad=True`), so `CHANNEL_WEIGHTS` only controls the relative
*magnitude* of each head's own update, not any cross-channel interference -- that interference
is removed structurally, not by loss weighting. Best checkpoint (lowest dev loss) is saved to
`artifacts/yolop_control_head_best.pt`.

In [ ]:
import os
from pathlib import Path

NUM_EPOCHS = 10
CHANNEL_WEIGHTS = torch.tensor([2.0, 1.0], device=device)  # steer, brake
USE_AMP = device == "cuda"  # mixed precision only makes sense on CUDA

Path("artifacts").mkdir(exist_ok=True)
# A distinct path from yolop_control_head_best.pt (the existing, already-validated baseline
# trained on raw, unmasked frames) -- this run trains on hard road/lane/vehicle-masked frames
# (section 4b) instead, and section 7's MAE comparison against the old checkpoint decides
# whether to promote this one, not an in-place overwrite.
CHECKPOINT_PATH = "artifacts/yolop_control_head_masked_best.pt"

# Full fresh run (random init, all 10 epochs, one continuous optimizer) -- not a resume.
# A previous attempt resumed from an epoch-8 checkpoint with a freshly-initialized AdamW
# optimizer (no momentum/variance state) at a 4x larger batch size for just 2 epochs; that
# discontinuity collapsed steer_head to a near-constant output even though aggregate dev_loss
# kept "improving" (steer/brake targets are both concentrated near zero, so a collapsed
# near-constant predictor can still score a deceptively low SmoothL1 loss). Training straight
# through avoids that discontinuity entirely.
optimizer = torch.optim.AdamW(control_model.trainable_parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = torch.nn.SmoothL1Loss(reduction="none")
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
best_dev_loss = float("inf")


def run_epoch(loader, train: bool):
    control_model.numeric_branch.train(train)
    control_model.shared_trunk.train(train)
    control_model.steer_head.train(train)
    control_model.brake_trunk.train(train)
    control_model.brake_head.train(train)
    total_loss, n = 0.0, 0
    for images, numeric, targets in loader:
        images = images.to(device, non_blocking=True)
        numeric = numeric.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with torch.set_grad_enabled(train):
            # The frozen YOLOP backbone forward is the dominant per-batch GPU cost (its own
            # forward already runs under no_grad inside YOLOPControlNet.forward), so autocast
            # mainly speeds that up; the ~94K-param trainable head is cheap either way.
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                preds = control_model(images, numeric)
                loss = (loss_fn(preds, targets) * CHANNEL_WEIGHTS).mean()
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * images.size(0)
        n += images.size(0)
    return total_loss / n


def save_checkpoint(path, dev_loss):
    torch.save({
        "numeric_branch": control_model.numeric_branch.state_dict(),
        "shared_trunk": control_model.shared_trunk.state_dict(),
        "steer_head": control_model.steer_head.state_dict(),
        "brake_trunk": control_model.brake_trunk.state_dict(),
        "brake_head": control_model.brake_head.state_dict(),
        "encoder_layer_index": control_model.ENCODER_LAYER_INDEX,
        "dev_loss": dev_loss,
    }, path)


for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    dev_loss = run_epoch(dev_loader, train=False)
    print(f"epoch {epoch:02d}  train_loss={train_loss:.4f}  dev_loss={dev_loss:.4f}")

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        # Versioned backup alongside the canonical path -- last session lost the epoch-8
        # checkpoint when epoch 9 overwrote CHECKPOINT_PATH in place, with no way to recover
        # it for comparison. Cheap insurance: these are ~1MB files (93K params).
        backup_path = f"artifacts/yolop_control_head_masked_epoch{epoch:02d}.pt"
        save_checkpoint(CHECKPOINT_PATH, dev_loss)
        save_checkpoint(backup_path, dev_loss)
        print(f"  saved new best checkpoint (dev_loss={dev_loss:.4f}) -> {CHECKPOINT_PATH} + {backup_path}")

# Free train_loader/dev_loader's persistent worker pools now -- neither is iterated
# again after this loop, and section 7c loads a second full model (baseline_model) +
# a 4th dataset/loader on a machine with only ~16GB system RAM. Leaving 12 idle worker
# processes (2 loaders x NUM_WORKERS=6) resident on top of that was the proximate cause
# of a hard kernel crash here previously (no Python traceback -- consistent with an
# OS-level OOM kill, not a catchable exception).
del train_loader, dev_loader

C:\Users\nalli\Documents\Workspace Uni\TP FINAL Vision\TP-final-Vision-Artificial\.venv\Lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4384.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


epoch 01  train_loss=0.0015  dev_loss=0.0009
  saved new best checkpoint (dev_loss=0.0009) -> artifacts/yolop_control_head_masked_best.pt + artifacts/yolop_control_head_masked_epoch01.pt
epoch 02  train_loss=0.0011  dev_loss=0.0008
  saved new best checkpoint (dev_loss=0.0008) -> artifacts/yolop_control_head_masked_best.pt + artifacts/yolop_control_head_masked_epoch02.pt
epoch 03  train_loss=0.0010  dev_loss=0.0008
epoch 04  train_loss=0.0010  dev_loss=0.0008
  saved new best checkpoint (dev_loss=0.0008) -> artifacts/yolop_control_head_masked_best.pt + artifacts/yolop_control_head_masked_epoch04.pt
epoch 05  train_loss=0.0010  dev_loss=0.0007
  saved new best checkpoint (dev_loss=0.0007) -> artifacts/yolop_control_head_masked_best.pt + artifacts/yolop_control_head_masked_epoch05.pt
epoch 06  train_loss=0.0009  dev_loss=0.0007
  saved new best checkpoint (dev_loss=0.0007) -> artifacts/yolop_control_head_masked_best.pt + artifacts/yolop_control_head_masked_epoch06.pt
epoch 07  train_loss

## 7. Evaluate on the test split

Reload the best checkpoint and report per-channel MAE both in normalized (BeamNG) units and
in human-interpretable units (steering degrees-equivalent, acceleration m/s²-equivalent) via
the inverse transforms in `comma_beamng_transforms.py`.

In [12]:
import numpy as np
from comma_beamng_transforms import beamng_controls_to_accel, beamng_steering_to_deg

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
control_model.numeric_branch.load_state_dict(checkpoint["numeric_branch"])
control_model.shared_trunk.load_state_dict(checkpoint["shared_trunk"])
control_model.steer_head.load_state_dict(checkpoint["steer_head"])
control_model.brake_trunk.load_state_dict(checkpoint["brake_trunk"])
control_model.brake_head.load_state_dict(checkpoint["brake_head"])

control_model.numeric_branch.eval()
control_model.shared_trunk.eval()
control_model.steer_head.eval()
control_model.brake_trunk.eval()
control_model.brake_head.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for images, numeric, targets in test_loader:
        preds = control_model(images.to(device), numeric.to(device))
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.numpy())

preds = np.concatenate(all_preds)
targets = np.concatenate(all_targets)
del test_loader  # done iterating; free its persistent worker pool before section 7c
mae = np.abs(preds - targets).mean(axis=0)

print("Per-channel MAE (normalized BeamNG units):")
print(f"  steer: {mae[0]:.4f}  (-1..1)")
print(f"  brake: {mae[1]:.4f}  (0..1)")

steer_deg_pred = beamng_steering_to_deg(preds[:, 0])
steer_deg_true = beamng_steering_to_deg(targets[:, 0])
# No throttle channel anymore -- pass throttle=0 to recover the brake-only longitudinal
# deceleration each brake value implies, for a human-readable m/s^2 sanity check.
decel_pred = beamng_controls_to_accel(np.zeros_like(preds[:, 1]), preds[:, 1])
decel_true = beamng_controls_to_accel(np.zeros_like(targets[:, 1]), targets[:, 1])

print("\nPer-channel MAE (human units):")
print(f"  steering angle:          {np.abs(steer_deg_pred - steer_deg_true).mean():.2f} deg")
print(f"  brake-equivalent decel:  {np.abs(decel_pred - decel_true).mean():.3f} m/s^2")

Per-channel MAE (normalized BeamNG units):
  steer: 0.0068  (-1..1)
  brake: 0.0355  (0..1)

Per-channel MAE (human units):
  steering angle:          2.28 deg
  brake-equivalent decel:  0.284 m/s^2


## 7b. Collapse check (not just aggregate MAE)

Aggregate MAE/dev_loss can look fine while a head has collapsed to predicting near its
target's marginal mean -- that's exactly what happened to `brake_head` originally (collapsed,
low-looking MAE) and to `steer_head` during a bad resume attempt (collapsed, dev_loss still
"improved"). Both `steer_target` and `brake_target` are concentrated near zero, which is what
lets a collapsed predictor hide from an aggregate metric. Checking each channel's prediction
*std* against its target's std, and the *correlation* between them, catches this directly: a
collapsed head has near-zero std and near-zero correlation regardless of what MAE says.

In [13]:
COLLAPSE_CORR_THRESHOLD = 0.15  # below this, a channel is flagged as likely collapsed

channel_names = ["steer", "brake"]
collapsed_any = False
for i, name in enumerate(channel_names):
    pred_std = preds[:, i].std()
    target_std = targets[:, i].std()
    corr = np.corrcoef(preds[:, i], targets[:, i])[0, 1]
    flag = " <-- COLLAPSED" if corr < COLLAPSE_CORR_THRESHOLD else ""
    if flag:
        collapsed_any = True
    print(f"{name}: pred_std={pred_std:.4f} target_std={target_std:.4f} corr={corr:.3f}{flag}")

if collapsed_any:
    print("\nWARNING: at least one channel looks collapsed despite the MAE above -- do not "
          "treat this checkpoint as fixed. Check the per-epoch backups in artifacts/ "
          "(yolop_control_head_epoch##.pt) for an earlier, non-collapsed epoch instead.")
else:
    print("\nNo collapse detected on either channel.")

steer: pred_std=0.0244 target_std=0.0562 corr=0.803
brake: pred_std=0.0272 target_std=0.0583 corr=0.588

No collapse detected on either channel.


## 7c. Compare against the pre-masking baseline checkpoint

`artifacts/yolop_control_head_best.pt` is the existing checkpoint trained on raw, unmasked
frames -- still untouched, since section 6 above saved to a different path
(`yolop_control_head_masked_best.pt`). Each checkpoint is evaluated on the test split it
actually matches end-to-end (old: raw frames `FRAMES_DIR`; new: masked frames
`MASKED_FRAMES_DIR`), since that mirrors how each one is actually used at inference time
(`2 Closed Loop BeamNG Control.ipynb`'s `LANE_MASK_MODE="off"` vs `"road"`). Promote the new
checkpoint (e.g. by copying it over `yolop_control_head_best.pt`) only if its MAE is not worse
here -- do not do this automatically.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [ ]:
OLD_CHECKPOINT_PATH = "artifacts/yolop_control_head_best.pt"

baseline_model = YOLOPControlNet(device=device).to(device)
old_ckpt = torch.load(OLD_CHECKPOINT_PATH, map_location=device)
baseline_model.numeric_branch.load_state_dict(old_ckpt["numeric_branch"])
baseline_model.shared_trunk.load_state_dict(old_ckpt["shared_trunk"])
baseline_model.steer_head.load_state_dict(old_ckpt["steer_head"])
baseline_model.brake_trunk.load_state_dict(old_ckpt["brake_trunk"])
baseline_model.brake_head.load_state_dict(old_ckpt["brake_head"])
baseline_model.eval()

raw_test_ds = Comma2k19ControlDataset(MANIFEST_PATH, FRAMES_DIR, split="test")
# num_workers=0 here, not loader_kwargs's 6 -- train_loader/dev_loader/test_loader's worker
# pools are already alive (persistent_workers=True, 3 x NUM_WORKERS=18 processes); adding a
# 4th persistent 6-worker pool on top of those blew past this machine's Windows shared-memory
# mapping quota (`Couldn't open shared file mapping ... error code: 1455`, the same ~16GB-RAM
# constraint already documented in section 5's NUM_WORKERS comment). This loader is iterated
# once for a one-off eval pass, so the multi-worker speedup isn't worth that risk here.
raw_test_loader = DataLoader(raw_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

old_preds, old_targets = [], []
with torch.no_grad():
    for images, numeric, targets in raw_test_loader:
        preds = baseline_model(images.to(device), numeric.to(device))
        old_preds.append(preds.cpu().numpy())
        old_targets.append(targets.numpy())

old_preds = np.concatenate(old_preds)
old_targets = np.concatenate(old_targets)
old_mae = np.abs(old_preds - old_targets).mean(axis=0)

print("Per-channel test MAE (normalized BeamNG units):")
print(f"  old (raw frames, internal soft mask)   steer={old_mae[0]:.4f}  brake={old_mae[1]:.4f}")
print(f"  new (hard-masked frames, this run)     steer={mae[0]:.4f}  brake={mae[1]:.4f}")
print()
steer_delta = mae[0] - old_mae[0]
brake_delta = mae[1] - old_mae[1]
print(f"delta (new - old): steer={steer_delta:+.4f}  brake={brake_delta:+.4f}  "
      f"(negative = new checkpoint is better)")
if steer_delta > 0 or brake_delta > 0:
    print("\nNew checkpoint is NOT strictly better on test MAE -- do not promote it over "
          "yolop_control_head_best.pt without further inspection (e.g. Grad-CAM quality in "
          "the closed loop, which was the actual motivation for masking).")
else:
    print("\nNew checkpoint matches or beats the old one on both channels.")

## 8. Inference sanity check

Spot-check a handful of test examples: predicted vs. ground-truth steer/brake.

In [ ]:
N_SAMPLES = 5
sample_idx = np.random.choice(len(test_ds), size=min(N_SAMPLES, len(test_ds)), replace=False)

for i in sample_idx:
    image, numeric, target = test_ds[i]
    with torch.no_grad():
        pred = control_model(image.unsqueeze(0).to(device), numeric.unsqueeze(0).to(device))[0].cpu().numpy()
    row = test_ds.df.iloc[i]
    print(f"{row['image_path']}")
    print(f"  current : throttle={numeric[0]:.2f} brake={numeric[1]:.2f} steer={numeric[2]:.2f} speed={numeric[3]:.1f}m/s")
    print(f"  target  : steer={target[0]:.2f} brake={target[1]:.2f}")
    print(f"  pred    : steer={pred[0]:.2f} brake={pred[1]:.2f}")
    print()

In [ ]:
# Save the notebook, then shut down the PC.
#
# NOTE on saving: this JS trick only works in classic Jupyter Notebook / JupyterLab
# frontends -- it does NOT reliably work in VS Code's notebook UI, which doesn't
# expose the same `Jupyter.notebook` JS object. If you're running this in VS Code,
# turn on "Files: Auto Save" (or just hit Ctrl+S yourself) instead of relying on
# this cell to persist your outputs.
from IPython.display import Javascript, display

display(Javascript("""
if (window.Jupyter && Jupyter.notebook) {
    Jupyter.notebook.save_checkpoint();
} else if (window.require) {
    require(['base/js/namespace'], function(Jupyter) { Jupyter.notebook.save_checkpoint(); });
}
"""))

import time
time.sleep(5)  # give the save a moment before shutting down

AUTO_SHUTDOWN = False  # set True for unattended overnight runs -- left off for interactive
                       # sessions (e.g. iterating on the brake_trunk fix) so this doesn't power
                       # off the machine while you're at the keyboard.
if AUTO_SHUTDOWN:
    import subprocess
    print("Shutting down in 60s. To cancel, open a terminal and run: shutdown /a")
    subprocess.run([
        "shutdown", "/s", "/t", "60",
        "/c", "Notebook run finished - shutting down (run 'shutdown /a' to cancel)",
    ])
else:
    print("AUTO_SHUTDOWN is False -- notebook finished, machine left running.")

## Next step (not done in this notebook)

`comma_beamng_transforms.py` and `artifacts/yolop_control_head_best.pt` are what
`2 Closed Loop BeamNG Control.ipynb` imports to actually drive: it reads `electrics['steering']`
plus the last applied throttle/brake as the numeric input, runs `YOLOPControlNet` for
`(steer, brake)` only, drives throttle from a separate non-adaptive cruise-control function
(never from this model), and sends both to `vehicle.control(steering=..., throttle=...,
brake=...)`. See that notebook for the closed loop itself, and `plan.md`/`NEXT_STEPS.md` for
the classical PID-based LKA/ACC alternative.